# Model Router — Runtime Demo: Reproducibility & Consistency

This notebook exercises the **runtime** routing path (`src/router/runtime/`) against the artifacts already built by the offline pipeline (`cluster-map.json` + `model-profiles.json`). It does not re-derive the pipeline itself — see the calibration/evaluation notebooks or `README.md` for that.

Goal: show, live and independently — not just assert — that:
1. the same input always produces the same routing decision (**reproducibility**),
2. the decision follows exactly the documented formula, with no hidden logic (**consistency**).

**Caveat to state up front, honestly:** `model-profiles.json` in this repo comes from a small, pipeline-completeness-focused calibration pass (a handful of tasks per cluster against `bigcodebench`/`ds1000` only — see the calibration notebook/README for the full discussion). Predicted error rates here are therefore noisy and often near-ceiling; that's a property of *this calibration run's scale*, not of the routing mechanism this notebook is checking.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))

import numpy as np

from router.common.artifacts import ARTIFACTS_DIR
from router.common.assign import assign_cluster
from router.common.config import load_calibration_config
from router.common.embedding import embed_one
from router.common.scoring import normalise_costs, static_price_per_1m
from router.runtime.context import load_routing_context
from router.runtime.decide import decide, decide_from_vector

CLUSTER_MAP_PATH = ARTIFACTS_DIR / "cluster-map.json"
MODEL_PROFILES_PATH = ARTIFACTS_DIR / "model-profiles.json"

DEMO_LAMBDA = 0.05  # a middle value from calibration.yaml's lambda_sweep; changed explicitly in the sweep section below
ctx = load_routing_context(CLUSTER_MAP_PATH, MODEL_PROFILES_PATH, DEMO_LAMBDA)

print(f"cluster map:    {ctx.cluster_map_id} (k={ctx.cluster_map.centroids.shape[0]}, dim={ctx.cluster_map.dimensions})")
print(f"model profiles: {ctx.profiles_id}")
print(f"embedding:      {ctx.embedding.model_id} ({ctx.embedding.dimensions}d)")
print(f"candidates:     {', '.join(sorted(m.model_id for m in ctx.candidates))}")
print(f"digest:         {ctx.digest}")

cluster map:    clustermap-2026-08-12-k24-s42 (k=24, dim=768)
model profiles: profiles-2026-08-12-cal-2026-08-12-125147
embedding:      jinaai/jina-embeddings-v2-base-code (768d)
candidates:     llama-3.1-8b-instant, llama-3.3-70b-versatile, openai/gpt-oss-120b
digest:         sha256:f9d49f0bfc83caed0807e26518fde1ab9368f7a8e812e1b763b90d94d242b48c


## 0. Coverage gaps

For any (model, cluster) pair with no calibration data, `score_candidates()` falls back to that model's **global** smoothed error rate (`common/scoring.py`'s `lookup_predicted_error`). This is correct, documented behaviour — not a bug — but the demo should say plainly which clusters are backed by real per-cluster signal today.

In [2]:
k = ctx.cluster_map.centroids.shape[0]
gaps = {
    m.model_id: [cid for cid in range(k) if str(cid) not in ctx.profiles_by_model[m.model_id].get("clusters", {})]
    for m in sorted(ctx.candidates, key=lambda m: m.model_id)
}
populated_clusters = sorted(set(range(k)) - set().union(*gaps.values())) if gaps else list(range(k))

for model_id, missing in gaps.items():
    print(f"{model_id}: {k - len(missing)}/{k} clusters have real calibration data")
print(f"\nclusters with data for every candidate (real per-cluster signal): {populated_clusters}")

llama-3.1-8b-instant: 9/24 clusters have real calibration data
llama-3.3-70b-versatile: 9/24 clusters have real calibration data
openai/gpt-oss-120b: 9/24 clusters have real calibration data

clusters with data for every candidate (real per-cluster signal): [0, 2, 3, 7, 9, 10, 13, 15, 20]


## 1. Reproducibility

Run the exact same prompt through `decide()` twice and diff every field of the returned `RoutingDecision` except wall-clock timing (`embed_ms`/`score_ms`, which naturally vary and aren't part of the decision itself). Everything else — cluster, scores, selection, and the `digest` — must match exactly.

In [5]:
DEMO_PROMPT = (
    "Write a Python function that reads a CSV file into a pandas DataFrame "
    "and returns the mean of each numeric column."
)

decision_a = decide(DEMO_PROMPT, ctx)
decision_b = decide(DEMO_PROMPT, ctx)

TIMING_FIELDS = {"embed_ms", "score_ms"}
mismatches = [
    f.name for f in decision_a.__dataclass_fields__.values()
    if f.name not in TIMING_FIELDS and getattr(decision_a, f.name) != getattr(decision_b, f.name)
]

assert not mismatches, f"reproducibility check FAILED on fields: {mismatches}"
print("✓ reproducible — every non-timing field matched across two independent runs")
print(f"  digest (both runs): {decision_a.digest}")
print(f"  cluster:  {decision_a.cluster.cluster_id} (distance={decision_a.cluster.distance:.4f})")
print(f"  selected: {decision_a.selected.model_id} (routing_score={decision_a.selected.routing_score:.4f}, error_source={decision_a.selected.error_source})")

✓ reproducible — every non-timing field matched across two independent runs
  digest (both runs): sha256:f9d49f0bfc83caed0807e26518fde1ab9368f7a8e812e1b763b90d94d242b48c
  cluster:  7 (distance=0.9181)
  selected: openai/gpt-oss-120b (routing_score=0.8264, error_source=cluster)


## 2. Cluster assignment

`assign_cluster()` picks the centroid with the smallest **squared Euclidean** distance (`common/assign.py`). Re-embed the demo prompt, compute that distance to all `k` centroids by hand, and confirm the nearest and runner-up match what the runtime returned above — exactly, not approximately.

In [6]:
vector = embed_one(DEMO_PROMPT, ctx.embedding)

distances = [
    (cid, float(np.dot(vector - ctx.cluster_map.centroids[cid], vector - ctx.cluster_map.centroids[cid])))
    for cid in range(k)
]
distances.sort(key=lambda pair: pair[1])
manual_best_id, manual_best_dist = distances[0]
manual_runner_up_id, manual_runner_up_dist = distances[1]

runtime_cluster = decision_a.cluster
assert manual_best_id == runtime_cluster.cluster_id, "manual nearest-centroid disagrees with assign_cluster()"
assert manual_runner_up_id == runtime_cluster.runner_up_cluster_id, "manual runner-up disagrees with assign_cluster()"
assert abs(manual_best_dist - runtime_cluster.distance) < 1e-9

print(f"✓ cluster assignment verified independently: cluster {manual_best_id}")
print("\nnearest 5 centroids by hand-computed squared distance:")
for cid, dist in distances[:5]:
    marker = " <- selected" if cid == manual_best_id else (" <- runner-up" if cid == manual_runner_up_id else "")
    print(f"  cluster {cid:>2}: {dist:.4f}{marker}")

✓ cluster assignment verified independently: cluster 7

nearest 5 centroids by hand-computed squared distance:
  cluster  7: 0.9181 <- selected
  cluster 15: 1.0454 <- runner-up
  cluster 10: 1.0459
  cluster  9: 1.1556
  cluster  8: 1.2248


## 3. Model selection

Spec §6:
```
routing_score[m] = predicted_error[m] + lambda * normalised_cost[m]
```
Pull `predicted_error` straight from `model-profiles.json` (falling back to the model's global rate exactly like `lookup_predicted_error` does) and `static_price` straight from `config/models.yaml`, then recompute the score for every candidate by hand and compare it to what `score_candidates()` actually returned.

In [5]:
cluster_id = decision_a.cluster.cluster_id
manual_rows = []
for model in ctx.candidates:
    profile = ctx.profiles_by_model[model.model_id]
    cluster_key = str(cluster_id)
    if cluster_key in profile.get("clusters", {}):
        predicted_error, error_source = profile["clusters"][cluster_key]["smoothed_error_rate"], "cluster"
    else:
        predicted_error, error_source = profile["global"]["smoothed_error_rate"], "model-global"
    manual_rows.append((model.model_id, predicted_error, error_source, static_price_per_1m(model)))

manual_normalised = normalise_costs([row[3] for row in manual_rows])
manual_scores = {
    model_id: predicted_error + DEMO_LAMBDA * norm_cost
    for (model_id, predicted_error, _source, _price), norm_cost in zip(manual_rows, manual_normalised)
}

runtime_scores = {s.model_id: s.routing_score for s in decision_a.scores}
for model_id, manual_score in manual_scores.items():
    assert abs(manual_score - runtime_scores[model_id]) < 1e-9, f"mismatch for {model_id}"

manual_winner = min(manual_scores, key=lambda m: (manual_scores[m], m))
assert manual_winner == decision_a.selected.model_id
print("✓ hand-recomputed routing_score matches score_candidates() for every candidate\n")

print(f"{'model_id':<24} {'predicted_error':<16} {'error_source':<14} {'normalised_cost':<17} {'routing_score':<10}")
for (model_id, predicted_error, error_source, _price), norm_cost in sorted(
    zip(manual_rows, manual_normalised), key=lambda pair: manual_scores[pair[0][0]]
):
    marker = "  <- selected" if model_id == manual_winner else ""
    print(f"{model_id:<24} {predicted_error:<16.4f} {error_source:<14} {norm_cost:<17.4f} {manual_scores[model_id]:<10.4f}{marker}")

✓ hand-recomputed routing_score matches score_candidates() for every candidate

model_id                 predicted_error  error_source   normalised_cost   routing_score
openai/gpt-oss-120b      0.8016           cluster        0.4960            0.8264      <- selected
llama-3.3-70b-versatile  0.7778           cluster        1.0000            0.8278    
llama-3.1-8b-instant     0.9603           cluster        0.0000            0.9603    


## 4. Consistency — the accuracy/cost trade-off across the full lambda sweep

A single decision matching the formula once is a weak guarantee. Sweeping `lambda` across `config/calibration.yaml`'s configured values and watching selection shift predictably — from the lowest-error candidate toward the cheapest as `lambda` grows — is a stronger one. Where the winner changes between two consecutive lambdas, the switch should land near the algebraic crossover point between those two candidates' scores.

In [6]:
calibration_config = load_calibration_config()

print(f"{'lambda':<8} {'selected':<24} {'routing_score':<14} {'error_source'}")
previous_winner = None
for lam in calibration_config.lambda_sweep:
    sweep_ctx = load_routing_context(CLUSTER_MAP_PATH, MODEL_PROFILES_PATH, lam)
    sweep_decision = decide_from_vector(vector, sweep_ctx)  # reuse the embedding computed above
    changed = " (changed)" if previous_winner is not None and sweep_decision.selected.model_id != previous_winner else ""
    print(
        f"{lam:<8} {sweep_decision.selected.model_id:<24} "
        f"{sweep_decision.selected.routing_score:<14.4f} {sweep_decision.selected.error_source}{changed}"
    )
    previous_winner = sweep_decision.selected.model_id

print("\ncheapest candidate is expected to win as lambda grows large; lambda=0 must match the lowest predicted_error candidate exactly")

lambda   selected                 routing_score  error_source
0        llama-3.3-70b-versatile  0.7778         cluster


0.02     llama-3.3-70b-versatile  0.7978         cluster
0.05     openai/gpt-oss-120b      0.8264         cluster (changed)


0.1      openai/gpt-oss-120b      0.8512         cluster
0.2      openai/gpt-oss-120b      0.9008         cluster


0.4      llama-3.1-8b-instant     0.9603         cluster (changed)

cheapest candidate is expected to win as lambda grows large; lambda=0 must match the lowest predicted_error candidate exactly


## 5. The fallback path

One prompt landing in a populated cluster (`error_source=cluster`, section 1–4 above) next to one deliberately chosen to be very different in kind — to illustrate the `model-global` fallback path actually firing, not just existing in code. Both are correct, documented behaviour.

In [4]:
CANDIDATE_PROMPTS = [
    "Explain the difference between a git rebase and a git merge in plain English.",
    "Write a bash one-liner that finds the 10 largest files under the current directory.",
    "Implement a binary search tree insert method in Rust.",
    "Summarize the plot of a mystery novel in three sentences.",
]

for prompt in CANDIDATE_PROMPTS:
    d = decide(prompt, ctx)
    sources = {s.model_id: s.error_source for s in d.scores}
    is_fallback_for_any = "model-global" in sources.values()
    label = "FALLBACK (>=1 candidate used global rate)" if is_fallback_for_any else "populated cluster"
    print(f"cluster {d.cluster.cluster_id:>2} | {label:<40} | selected={d.selected.model_id} | {prompt[:60]}")

cluster 13 | populated cluster                        | selected=openai/gpt-oss-120b | Explain the difference between a git rebase and a git merge 
cluster 13 | populated cluster                        | selected=openai/gpt-oss-120b | Write a bash one-liner that finds the 10 largest files under


selected openai/gpt-oss-120b via its global error rate — cluster 12 has no calibration coverage for this model


cluster 12 | FALLBACK (>=1 candidate used global rate) | selected=openai/gpt-oss-120b | Implement a binary search tree insert method in Rust.
cluster  7 | populated cluster                        | selected=openai/gpt-oss-120b | Summarize the plot of a mystery novel in three sentences.


## Summary

- **Reproducibility**: two independent calls to `decide()` with the same prompt + lambda produced byte-identical decisions, including the artifact digest.
- **Consistency (assignment)**: the cluster the runtime picked matched an independently hand-computed nearest-centroid calculation exactly.
- **Consistency (selection)**: the routing formula (`error + lambda * normalised_cost`) re-derived by hand from the raw artifacts matched `score_candidates()`'s output exactly, for every candidate.
- **Consistency (trade-off)**: sweeping lambda shifted selection from lowest-error toward cheapest in the expected direction.
- **Honesty about scope**: coverage gaps and the model-global fallback were shown explicitly, not glossed over — this calibration pass covers 9/24 clusters with real per-cluster signal today.

None of this depends on the specific numbers in the current `model-profiles.json` — re-run this notebook after any future `calibrate` run and it re-verifies against whatever artifact is on disk.